# Tutorial Passo 1: Download dos dados do GWOSC

Antes de começarmos, vamos delinear o conteúdo dos tutoriais:
- O Notebook 1 explica como baixar dados do site do GWOSC. 
- O Notebook 2 apresenta detalhes sobre a leitura do conteúdo de um arquivo. 
- O Notebook 3 apresenta indicadores de qualidade, um conceito importante ao trabalhar com dados de ondas gravitacionais. 
- O Notebook 4 apresenta outro conceito importante para dados de ondas gravitacionais: o domínio da frequência. 
- O Notebook 5 apresenta uma interface de nível mais alto que oculta muitos detalhes para proporcionar um acesso mais fácil aos dados.

Se você quiser entender os detalhes sobre os arquivos GWOSC, basta executar os tutoriais na ordem indicada.
Se você se sente confortável com Python e não quer se preocupar com os detalhes, vá diretamente para [notebook 5](<05 - GWpy Examples.ipynb>).

Este tutorial mostrará como baixar dados do [GWOSC website](https://gwosc.org).

## Explorar conjuntos de dados disponíveis

Vá para o [GWOSC website](https://gwosc.org/) e abra "Get Data" na barra de menus e, em seguida, clique em "Data Set".
A página [Observatory Data Sets](https://gwosc.org/data/) exibirá a lista de conjuntos de dados.

## Use timelines (linhas do tempo) para mostrar os períodos de dados disponíveis.

Os dados dos detectores LVK estão disponíveis publicamente apenas nos períodos em que os detectores operavam em condições normais, identificados pela flag DATA (ou seja, dados disponíveis naquele momento).
As timelines (linhas do tempo) do GWOSC oferecem uma ferramenta gráfica prática para identificar os períodos em que os detectores estavam coletando dados.

Vamos começar analisandoa timeline de dados do H1 durante a segunda parte da terceira campanha de observação (O3b):
  - Na página [Observatory Data Sets](https://gwosc.org/data/), encontre: "O3b" Data Release.
  - Clique no [ícone da Timeline](https://gwosc.org/timeline/show/O3b_16KHZ_R1/H1_DATA*L1_DATA*V1_DATA/1256655618/12708000/). Você deve ver gráficos como estes: ![timeline](./img/timeline.png)

Aqui estão algumas observações para ajudar a entender as timelines do GWOSC:
- O rótulo na extremidade esquerda indica qual instrumento está representado no gráfico. Neste exemplo, H1 corresponde ao detector "Hanford One". 
- A curva mostra se o detector estava coletando dados naquele momento específico: uma barra vertical indica que o detector está coletando dados, enquanto a ausência de barra indica que o detector não estava coletando dados. 
- Este conjunto de dados abrange a segunda parte da terceira campanha de observação (O3b), ou seja, de 01/11/2019 às 15:00:00 UTC (GPS=1256655618) até 27/03/2020 às 17:00:00 UTC (GPS=1269363618). 
- Você pode usar o mouse para aplicar zoom e mover-se pelos gráficos, bem como baixar a figura como imagem para o seu computador clicando no ícone "..." no canto superior direito do gráfico.

## Baixe arquivos de dados do arquivo de dados

Agora que temos uma noção de quando os dados estão disponíveis, vamos obter os arquivos de dados propriamente ditos.
Vamos tentar encontrar dados referentes ao mês de janeiro de 2020.
Podemos consultar o arquivo da O3b em busca dos dados desejados:

- Clique no link "Get Data" na barra de menu e, em seguida, em "Data Sets". 
- Em "O3b Data Release", clique no ícone "4 kHz Data":

- Use os botões de opção (*radio buttons*) para selecionar "H1". 
- No formulário web, insira as datas 2020-01-01T00:00:00 UTC em "Start Time" e verifique o tempo GPS (GPS=1261872018) e insira em "End Time" 2020-01-31T23:59:59 UTC e verifique o tempo GPS (GPS=1264550417).
- Escolha o formato de arquivo de saída: "Time series data in HDF5 and Frame files"
- Em seguida, clique no botão de continuar.

Como você pode ver, usamos a hora GPS.
Para encontrar a hora GPS de um horário UTC específico (ou vice-versa), você pode usar a [ferramenta de conversão do tempo](https://gwosc.org/gps/).

Isso consultará o banco de dados em busca de arquivos de dados referentes ao período entre as datas informadas.
Você deverá ver uma lista semelhante a esta: ![archive content](./img/archive_content.png)

Cada linha da tabela corresponde a um arquivo de dados (ou *tile*) que abrange 4096 segundos de tempo de calendário.
Um determinado instrumento pode estar operacional ou inoperante durante qualquer fração desse tempo, e a coluna da extrema direita indica qual porcentagem dos 4096 segundos contém dados do modo científico.

Vamos baixar um arquivo de dados que utilizaremos durante o restante deste tutorial.
Como queremos um arquivo contendo principalmente dados do "Science Mode" (Modo Científico), vamos baixar o arquivo que começa no instante GPS 1264312320.

Para baixar o arquivo, basta clicar no link na coluna sob o cabeçalho "HDF5".
O arquivo baixado deve ser nomeado `H-H1_GWOSC_O3b_4KHZ_R1-1264312320-4096.hdf5`.

<div class="alert alert-block alert-warning">
<div><b>&#9888; Warning</b></div>
    Após a conclusão do download, mova o arquivo para o diretório onde os notebooks estão armazenados. 
Se você utilizar o Binder ou o Colab para executar os notebooks, faça o upload deste arquivo para a instância do Binder/Colab. 
Alternativamente, a próxima seção mostrará como baixar esse arquivo programaticamente na pasta atual.
</div>

## Download programático

É possível interagir com o site do GWOSC utilizando código Python.
Ao criar requisições web específicas, é possível obter as informações em um formato utilizável pelo programa.
Você talvez tenha reconhecido o que se chama de API web (se não estiver familiarizado com esse conceito, não se preocupe com isso por enquanto).

No exemplo a seguir, mostraremos como baixar o mesmo arquivo usando requisições web.
Para isso, utilizamos o [pacote Python requests](https://requests.readthedocs.io/en/latest/) o que permite enviar solicitações ao servidor de maneira simples.

Primeiro, vamos definir uma função que retornará a lista de arquivos de *strain* entre dois tempos GPS para uma determinada campanha de observação e um detector específico.

In [3]:
# Importa o pacote
import requests

# Definie a função
def fetch_strain_list(run, detector, gps_start, gps_end):
    "Retorne a lista do arquivo de strain (deformação) para `run` e `detector`."

    # Obtenha a lista de strain em (https://gwosc.org/archive/links/O3b_4KHZ_R1/H1/1261872018/1264550417/)
    fetch_url = (
        f"https://gwosc.org/archive/links/"
        f"{run}/{detector}/{gps_start}/{gps_end}/json/"
    )
    response = requests.get(fetch_url)
    response.raise_for_status()
    return response.json()["strain"]

Agora, vamos usar isso com os mesmos tempos de GPS de antes:

In [4]:
# Chama a função e retorna a lista de strain

strain_files = fetch_strain_list("O3b_4KHZ_R1", "H1", 1261872018, 1264550417)
print(f"Found {len(strain_files)} files")
print(strain_files[0:5])

Found 1062 files
[{'GPSstart': 1261965312, 'UTCstart': '2020-01-02T01:54:54', 'detector': 'H1', 'sampling_rate': 4096, 'duration': 4096, 'format': 'hdf5', 'url': 'https://gwosc.org/archive/data/O3b_4KHZ_R1/1261436928/H-H1_GWOSC_O3b_4KHZ_R1-1261965312-4096.hdf5', 'min_strain': -5.393283677409768e-19, 'max_strain': 4.920883533490361e-19, 'mean_strain': 5.0291205486927025e-25, 'stdev_strain': 9.167596975962463e-20, 'duty_cycle': 91.2353515625, 'BLRMS200': 5.472092079816551e-24, 'BLRMS1000': 1.9148069663502894e-21, 'BNS': 106.15558044159869}, {'GPSstart': 1261965312, 'UTCstart': '2020-01-02T01:54:54', 'detector': 'H1', 'sampling_rate': 4096, 'duration': 4096, 'format': 'gwf', 'url': 'https://gwosc.org/archive/data/O3b_4KHZ_R1/1261436928/H-H1_GWOSC_O3b_4KHZ_R1-1261965312-4096.gwf', 'min_strain': -5.393283677409768e-19, 'max_strain': 4.920883533490361e-19, 'mean_strain': 5.0291205486927025e-25, 'stdev_strain': 9.167596975962463e-20, 'duty_cycle': 91.2353515625, 'BLRMS200': 5.472092079816551e

Reserve um tempo para inspecionar a variável `strain_files`.
Você verá que ela contém os mesmos dados da tabela acima.

Para localizar o arquivo HDF5 que começa no tempo GPS 1264312320, podemos proceder da seguinte forma:

In [5]:
def download_strain_file(download_url):
    "Baixe o arquivo do strain na URL fornecida e salve-o no disco." 
    # Na próxima linha, extraio o nome do arquivo da URL de download. 
    # O ideal seria obter o nome do arquivo a partir do
    # cabeçalho de resposta Content-Disposition.
    
    filename = download_url.split("/")[-1]
    with requests.get(download_url, stream=True) as r:
        r.raise_for_status()
        with open(filename, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    return filename


for a_file in strain_files:
    if a_file["GPSstart"] == 1264312320 and a_file["format"] == "hdf5":
        print(f"Downloading {a_file['url']}")
        fname = download_strain_file(a_file["url"])

O arquivo deve ser baixado na pasta atual.

<div class="alert alert-block alert-warning">
<div><b>&#9888; Warning</b></div>
    Você pode se sentir tentado a usar esse tipo de código para baixar dados em massa, percorrendo todos os arquivos em um loop. 
    No entanto, isso sobrecarregaria os servidores do GWOSC e provavelmente falharia devido a diversas instabilidades na rede. 
    Portanto, lembre-se de que grandes poderes trazem grandes responsabilidades e não faça isso. 
    Se você quiser baixar uma grande quantidade de dados, dê uma olhada em <a href="https://computing.docs.ligo.org/guide/compsoft/roadmap/LVK/cvmfs_for_data/?h=cvmfs">CVMFS</a>.
</div>

## Qual é o próximo passo?

No [próximo passo](<02 - What's in a GWOSC Data File.ipynb>), você aprenderá como os dados de ondas gravitacionais são armazenados neste arquivo.